# PREPROCESSING

## Data Loading

In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def load_dataset(folder):
    all_files = glob.glob(f"{folder}/**/*.csv", recursive=True)
    samples = []
    
    for filepath in all_files:
        df = pd.read_csv(filepath)
        file_id = df['file_id'].iloc[0]
        label = df['label'].iloc[0] if 'label' in df.columns else None
        
        features = {}
        for col in ['mean_x','mean_y','mean_z','std_x','std_y','std_z']:
            features[f'{col}_mean']   = df[col].mean()
            features[f'{col}_std']    = df[col].std()
            features[f'{col}_min']    = df[col].min()
            features[f'{col}_max']    = df[col].max()
            features[f'{col}_median'] = df[col].median()
        
        features['file_id'] = file_id
        features['label']   = label
        samples.append(features)
    
    return pd.DataFrame(samples)

train_df = load_dataset("train")
test_df  = load_dataset("test")
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Train: (11020, 32), Test: (6849, 32)


## Preparation of X,Y and normalization

In [4]:
feature_cols = [c for c in train_df.columns 
                if c not in ['file_id','label']]

X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_test  = test_df[feature_cols].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (11020, 30)
X_test shape:  (6849, 30)


## Random Forest model

In [5]:
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

scores = cross_val_score(rf, X_train, y_train, 
                         cv=5, scoring='f1_macro')
print(f"Random Forest F1-macro: {scores.mean():.4f}")

Random Forest F1-macro: 0.6268


## XGBoost

In [6]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)

scores = cross_val_score(xgb, X_train, y_train,
                         cv=5, scoring='f1_macro')
print(f"XGBoost F1-macro: {scores.mean():.4f}")

XGBoost F1-macro: 0.6661


## XGBoost V2

In [ ]:
def load_dataset_v2(folder):
    all_files = glob.glob(f"{folder}/**/*.csv", recursive=True)
    samples = []
    
    for filepath in all_files:
        df = pd.read_csv(filepath)
        file_id = df['file_id'].iloc[0]
        label = df['label'].iloc[0] if 'label' in df.columns else None
        
        features = {}
        for col in ['mean_x','mean_y','mean_z','std_x','std_y','std_z']:
            features[f'{col}_mean']     = df[col].mean()
            features[f'{col}_std']      = df[col].std()
            features[f'{col}_min']      = df[col].min()
            features[f'{col}_max']      = df[col].max()
            features[f'{col}_median']   = df[col].median()
            features[f'{col}_skew']     = df[col].skew()        # šikmost
            features[f'{col}_kurt']     = df[col].kurt()        # špičatost
            features[f'{col}_q25']      = df[col].quantile(0.25)# 1. kvartil
            features[f'{col}_q75']      = df[col].quantile(0.75)# 3. kvartil
            features[f'{col}_range']    = df[col].max() - df[col].min() # rozsah
        
        # celkové zrychlení (magnitude)
        df['magnitude'] = np.sqrt(df['mean_x']**2 + df['mean_y']**2 + df['mean_z']**2)
        features['magnitude_mean'] = df['magnitude'].mean()
        features['magnitude_std']  = df['magnitude'].std()
        features['magnitude_max']  = df['magnitude'].max()
        
        features['file_id'] = file_id
        features['label']   = label
        samples.append(features)
    
    return pd.DataFrame(samples)

train_df2 = load_dataset_v2("train")
test_df2  = load_dataset_v2("test")

feature_cols2 = [c for c in train_df2.columns 
                if c not in ['file_id','label']]

X_train2 = scaler.fit_transform(train_df2[feature_cols2].values)
X_test2  = scaler.transform(test_df2[feature_cols2].values)

xgb2 = XGBClassifier(n_estimators=200, learning_rate=0.1, 
                      random_state=42, eval_metric='mlogloss')

scores2 = cross_val_score(xgb2, X_train2, y_train,
                          cv=5, scoring='f1_macro')
print(f"XGBoost v2 (more features) F1-macro: {scores2.mean():.4f}")



NameError: name 'scaler' is not defined

## XGBoost v3

In [5]:
# =========================
# IMPORTS
# =========================

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import numpy as np


# =========================
# CROSS VALIDATION
# =========================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# =========================
# XGBOOST
# =========================

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist'
)


# =========================
# LIGHTGBM
# =========================

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
print('submission.csv vytvořen')

ModuleNotFoundError: No module named 'catboost'